In [ ]:
!npm install -g localtunnel


In [ ]:
import os
os.chdir('/content')
import subprocess
print('Installing the isolated, approval-gated terminal runtime...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'bubblewrap', 'util-linux'], check=True)
os.environ['OPENSWARM_ENABLE_SANDBOX'] = '1'
!rm -rf /content/venv
!pip install virtualenv
!virtualenv /content/venv
!rm -rf /content/usman-and-umar-app
!git clone --single-branch --branch feature/hybrid-task-orchestrator https://github.com/saedhawwa23-rgb/usman-and-umar-app.git /content/usman-and-umar-app
!/content/venv/bin/pip install -r /content/usman-and-umar-app/cloud-backend/requirements.txt
print("Backend dependencies and repository are ready.")


### Upstash Redis setup (standalone connection check)

Create a Redis database in Upstash and copy its **REST URL** and **REST token**. In Colab **Secrets**, add `UPSTASH_REDIS_REST_URL` and `UPSTASH_REDIS_REST_TOKEN`, then enable notebook access. Also configure `APP_CLIENT_TOKEN` and the optional model/GitHub secrets as needed. The next cell checks Redis with a temporary `SET`/`GET` round-trip and removes its probe key. The task, checkpoint, lease, and receipt document is stored at `task_{task_id}`.


In [ ]:
import os, sys, uuid
from pathlib import Path
from google.colab import userdata

SECRET_NAMES = (
    "UPSTASH_REDIS_REST_URL", "UPSTASH_REDIS_REST_TOKEN", "APP_CLIENT_TOKEN",
    "NVIDIA_API_KEY", "OPENROUTER_API_KEY", "OPENROUTER_MODEL", "GITHUB_TOKEN", "TUNNEL_SUBDOMAIN",
)
for name in SECRET_NAMES:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if isinstance(value, str) and value.strip():
        os.environ[name] = value.strip()
    elif name not in os.environ:
        os.environ.pop(name, None)

if not os.environ.get("UPSTASH_REDIS_REST_URL") or not os.environ.get("UPSTASH_REDIS_REST_TOKEN"):
    raise RuntimeError("NOT READY — add UPSTASH_REDIS_REST_URL and UPSTASH_REDIS_REST_TOKEN to Colab Secrets.")
if not os.environ.get("APP_CLIENT_TOKEN"):
    raise RuntimeError("NOT READY — add APP_CLIENT_TOKEN to Colab Secrets and configure the same value in the Android app.")

backend_dir = Path("/content/usman-and-umar-app/cloud-backend")
if not backend_dir.is_dir():
    raise RuntimeError("Backend repository is missing; run the setup cell first.")
sys.path.insert(0, str(backend_dir))
from redis_task_storage import RedisTaskStorage
RedisTaskStorage().probe()

_runtime_nonce = uuid.uuid4().hex
os.environ["OPENSWARM_SERVER_ID"] = f"colab-{_runtime_nonce[:12]}"
os.environ["OPENSWARM_INSTANCE_ID"] = _runtime_nonce
print("Upstash Redis SET/GET preflight passed; configured secret values were not displayed.")


In [ ]:
# Install Playwright Chromium into a stable Colab path and verify it can launch before uvicorn.
import os, subprocess
from pathlib import Path

venv_python = Path("/content/venv/bin/python")
venv_pip = Path("/content/venv/bin/pip")
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "/content/ms-playwright"
if not venv_python.is_file() or not venv_pip.is_file():
    raise RuntimeError("Playwright setup failed: /content/venv is missing; run the repository/dependency setup cell first.")
browser_path = Path(os.environ["PLAYWRIGHT_BROWSERS_PATH"])
browser_path.mkdir(parents=True, exist_ok=True)

def run_checked(label, command):
    print(f"{label}...")
    try:
        subprocess.run(command, check=True, env=os.environ.copy())
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"{label} failed with exit code {exc.returncode}; uvicorn must not be started.") from exc

# Playwright is in cloud-backend/requirements.txt; repair only if that install was incomplete.
playwright_check = subprocess.run(
    [str(venv_python), "-c", "import playwright"],
    env=os.environ.copy(), check=False,
)
if playwright_check.returncode != 0:
    run_checked(
        "Installing the backend's Playwright package into /content/venv",
        [str(venv_pip), "install", "playwright>=1.51.0,<2"],
    )

run_checked(
    "Installing Chromium into /content/ms-playwright (without apt/--with-deps)",
    [str(venv_python), "-m", "playwright", "install", "chromium"],
)
smoke_check = r'''import asyncio
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        print(f"Chromium launch smoke check passed (version {browser.version}).")
        await browser.close()

asyncio.run(main())
'''
run_checked(
    "Launching the installed Chromium smoke check",
    [str(venv_python), "-c", smoke_check],
)
print(f"Playwright browser setup is ready; PLAYWRIGHT_BROWSERS_PATH={browser_path}.")


In [ ]:
# Start FastAPI only after the Upstash Redis preflight has succeeded.
# The subprocess runs the same command as: uvicorn main:app --host 0.0.0.0 --port 7860.
import os, subprocess, sys, time
from pathlib import Path
backend_dir = Path('/content/usman-and-umar-app/cloud-backend')
if not backend_dir.is_dir():
    raise RuntimeError("Backend folder missing. Repository clone failed; server will not be started.")
os.chdir(backend_dir)
backend_log_path = Path('/content/fastapi.log')
backend_log = open(backend_log_path, 'w', buffering=1)
try:
    fastapi_process = subprocess.Popen(
        ['/content/venv/bin/python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'],
        cwd=str(backend_dir), stdout=backend_log, stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
except Exception as exc:
    backend_log.flush()
    backend_log.close()
    print(f"FAILED to launch FastAPI/uvicorn: {type(exc).__name__}: {exc}")
    if backend_log_path.exists():
        print("FastAPI startup log (last 80 lines):")
        print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-80:]) or '<empty>')
    raise RuntimeError("Could not start uvicorn on 0.0.0.0:7860; see the startup diagnostics above.") from exc
print(f"FastAPI process started (PID {fastapi_process.pid}) on 0.0.0.0:7860; log: {backend_log_path}")
print("Waiting for Redis task-state/checkpoint/lease readiness.")


In [ ]:
# Require the backend's real readiness contract. A legacy {"status":"ok"} response is NOT ready.
import requests, time

health_url = 'http://127.0.0.1:7860/health'
required_true = ('ready', 'auth_ready', 'redis_ready', 'checkpoints_ready', 'checkpoint_storage_ready', 'lease_ready', 'lease_coordinator_ready', 'accepting_new_tasks')
required_false = ('draining',)
deadline = time.monotonic() + 180
last_health = None
health_ok = False
last_error = 'No health response received yet.'
while time.monotonic() < deadline:
    if fastapi_process.poll() is not None:
        last_error = f"FastAPI exited with code {fastapi_process.returncode}."
        try:
            backend_log.flush()
            print("FastAPI process exited; startup log (last 100 lines):")
            print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-100:]) or '<empty>')
        except Exception as log_exc:
            print(f"Could not read FastAPI startup log: {type(log_exc).__name__}: {log_exc}")
        break
    try:
        response = requests.get(health_url, timeout=5)
        last_health = response.json() if response.headers.get('content-type', '').startswith('application/json') else None
        if response.status_code == 200 and isinstance(last_health, dict):
            missing_or_false = [key for key in required_true if last_health.get(key) is not True]
            missing_or_true = [key for key in required_false if last_health.get(key) is not False]
            if not missing_or_false and not missing_or_true:
                health_ok = True
                print("Backend health is ready: Redis, checkpoints, lease, auth, and accepting-new-tasks readiness are all true.")
                break
            last_error = f"Health has not declared readiness; false/missing checks: {missing_or_false}; true/missing drain flags: {missing_or_true}; response: {last_health}"
        else:
            last_error = f"Health returned HTTP {response.status_code}: {last_health or response.text[:500]}"
    except Exception as exc:
        last_error = f"Health request failed: {type(exc).__name__}: {exc}"
    time.sleep(3)
else:
    missing_or_false = [key for key in required_true if not isinstance(last_health, dict) or last_health.get(key) is not True]
    missing_or_true = [key for key in required_false if not isinstance(last_health, dict) or last_health.get(key) is not False]

if not (health_ok and isinstance(last_health, dict)
        and all(last_health.get(key) is True for key in required_true)
        and all(last_health.get(key) is False for key in required_false)):
    if fastapi_process.poll() is None:
        fastapi_process.terminate()
        try:
            fastapi_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            fastapi_process.kill()
    print(f"NOT READY — tunnel will not be launched. Last health result: {last_error}")
    try:
        backend_log.flush()
        print("FastAPI log (last 100 lines):")
        print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-100:]) or '<empty>')
    except Exception as log_exc:
        print(f"Could not read FastAPI log: {type(log_exc).__name__}: {log_exc}")
    raise RuntimeError("FastAPI did not report ready, auth_ready, redis_ready, checkpoints_ready, lease_ready, and accepting_new_tasks=true.")


### Configure a per-server Localtunnel subdomain

In Google Colab, open **Secrets** (key icon), add a secret named `TUNNEL_SUBDOMAIN`, enter a unique value such as `usman-umar-1` for Server 1 or `usman-umar-2` for Server 2, and enable notebook access. If unset, the tunnel cell defaults to `usman-umar-1`. The tunnel starts only after the health gate above reports fully ready.


In [ ]:
# Tunnel launch is deliberately after successful Redis/checkpoint/lease health readiness.
import os, re, subprocess, time
try:
    from google.colab import userdata
    _tunnel_secret = userdata.get("TUNNEL_SUBDOMAIN")
    if isinstance(_tunnel_secret, str) and _tunnel_secret.strip():
        os.environ["TUNNEL_SUBDOMAIN"] = _tunnel_secret.strip()
except Exception:
    pass  # Environment/default fallback; the secret itself is never printed.
subdomain = os.environ.get("TUNNEL_SUBDOMAIN", "usman-umar-1").strip()
if not re.fullmatch(r"[a-zA-Z0-9-]{1,63}", subdomain):
    raise RuntimeError("TUNNEL_SUBDOMAIN contains invalid characters; tunnel was not launched.")
tunnel_log = open('/content/localtunnel.log', 'w')
tunnel_process = subprocess.Popen(
    ['npx', '--yes', 'localtunnel', '--port', '7860', '--subdomain', subdomain],
    stdout=tunnel_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
time.sleep(3)
if tunnel_process.poll() is not None:
    tunnel_log.flush()
    print("Localtunnel exited before becoming available; inspect /content/localtunnel.log. No URL is being published.")
    raise RuntimeError(f"Localtunnel exited with code {tunnel_process.returncode}.")
print(f"Public URL: https://{subdomain}.loca.lt")
print("Open the URL in Chrome first to bypass the Localtunnel warning page.")


In [ ]:
# Includes langchain-openai, required by OpenSwarm's ChatOpenAI model adapter.
import subprocess
_ai_log = open('/content/pip_ai.log', 'w')
ai_install_process = subprocess.Popen(
    ['/content/venv/bin/pip', 'install', '-r', '/content/usman-and-umar-app/cloud-backend/requirements-ai.txt'],
    stdout=_ai_log, stderr=subprocess.STDOUT,
)
print("Downloading OpenSwarm AI packages in the background. AI tasks will work once installation completes.")


In [ ]:
import importlib.util, subprocess, sys, time
ready = False
for _ in range(6):
    if ai_install_process.poll() not in (None, 0):
        print("OpenSwarm AI package installation failed; inspect /content/pip_ai.log.")
        raise RuntimeError(f"AI package installer exited with code {ai_install_process.returncode}.")
    check = subprocess.run(['/content/venv/bin/python', '-c', "import importlib.util,sys; sys.exit(0 if importlib.util.find_spec('langchain') else 1)"], check=False)
    if check.returncode == 0:
        print('OpenSwarm AI Engine is ready!')
        ready = True
        break
    time.sleep(5)
if not ready:
    print('OpenSwarm AI packages are still installing in the background; inspect /content/pip_ai.log if this persists.')


## Drain limitation

Colab does not provide this notebook with a dependable runtime shutdown/drain hook. Runtime expiration, disconnection, or forced termination can stop FastAPI without a graceful task drain; do not assume an in-flight task is checkpointed before shutdown. Redis persistence is best-effort: task updates use GET followed by SET and are not atomic across runtimes. A successful readiness round-trip does not prove exclusive lease ownership, cross-server compare-and-swap, or checkpoint durability after abrupt runtime loss.
